In [6]:
import os
%load_ext autoreload
%autoreload 1
%aimport utility.functions
%aimport src.analysis
%aimport src.dataset
import numpy as np
import utility.functions as fn
import src.analysis as an
import src.dataset as ds
import torch
from scipy.stats import norm
import pandas as pd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [44]:
channels = [3, 5, 9, 11, 13, 15, 17, 19]
signal_amps = [0.00231, 0.00180, 0.00126, 0.00130, 0.00179, 0.00134, 0.001139, 0.00112]
channel_ID = [5, 9, 11, 10, 4, 2, 12, 3]

# Initialize output DataFrame
out_data = pd.DataFrame(columns=["channel_ID", "channel", "acceptance", "BI", "rp"])
out_data.to_csv('../outputs/BI_results.csv', index=False)

# Iterate over sorted channels
for index_channel in np.argsort(channel_ID):
    channel = channels[index_channel]
    signal_amp = signal_amps[index_channel]

    # Load datasets and channel specifications
    data_single, data_pileup, _, _, _ = ds.create_data_sets(channel, win_length=2000, shift=-2, pos_prefix="_bis", differenciate=True)
    H_unit, S, bslspec, w = ds.get_channel_specs(channel, N_deriv=1)

    # Convert data to PyTorch tensors
    S_torch = torch.tensor(S, dtype=torch.cfloat)
    H_unit_torch = torch.tensor(H_unit, dtype=torch.cfloat)
    w_torch = torch.tensor(w, dtype=torch.cfloat)
    t_torch = torch.linspace(0, 8e-4, 200, dtype=torch.cfloat)
    bslspec_torch = torch.tensor(bslspec, dtype=torch.cfloat)
    signal_amp_torch = torch.tensor(signal_amp, dtype=torch.float32)

    # Loop over acceptance levels
    for acceptance in [0.7, 0.8, 0.84, 0.87, 0.9, 0.92, 0.93, 0.94, 0.95, 0.96, 0.99]:
        N_sigma = norm.ppf(1 - (1 - acceptance) * 100 / 100)
        f1_opt, f2_opt, _ = an.optimize_filters(S_torch, H_unit_torch, w_torch, t_torch, bslspec_torch, signal_amp_torch, N_sigma=N_sigma, n_trials=1000)
        
        #save_filter_function 
        np.save(f"../outputs/Pileup_filter_functions/channel_{channel}/functions_eff{int(acceptance*100):d}", np.array([f1_opt, f2_opt]))
        
        # Compute PSD and BI
        PSD_pileup = an.get_PSD(data_pileup, H_unit, f1_opt, f2_opt)
        PSD_single = an.get_PSD(data_single, H_unit, f1_opt, f2_opt)
        cut = np.percentile(PSD_single, 100 - acceptance * 100)
        rp = np.mean(PSD_pileup < cut)
        BI = fn.calculate_BI(rp, 0.8)

        # Append results to DataFrame
        out_data = pd.concat([out_data, pd.DataFrame([{
            "channel_ID": channel_ID[index_channel],
            "channel": channel,
            "acceptance": acceptance,
            "BI": BI,
            "rp": rp
        }])], ignore_index=True)

# Save results to CSV
out_data.to_csv('../outputs/BI_results.csv', index=False)